In [1]:
print("Hola")

Hola


In [2]:
import os
import sys
print(os.getcwd())

sys.path.append(r'c:\Users\jmass\Documents\GitHub\TFG_GEINF\src\data')
from DatasetGenerator import DatasetGenerator


c:\Users\jmass\Documents\GitHub\TFG_GEINF\notebooks\experiments\params_prediction


In [3]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [4]:
data_raw = dg.generate_RandomWalk(random_params=True, num_passos=200)

In [5]:
data_raw["attributes"][0]["movement_type"]

'hard_stop'

In [6]:
df = data_raw[["coords", "attributes"]].copy()
df["movement_type"] = df["attributes"].apply(lambda x: x["movement_type"])
df = df[["coords", "movement_type"]]

In [7]:
data = df
data

,coords,movement_type
0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, -1....",hard_stop
1,"[[0.0, 0.0, -1.0, -1.0, -1.0, -2.0, -2.0, -1.0...",nonstop
2,"[[0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0,...",hard_stop
3,"[[0.0, -1.0, -1.0, -2.0, -1.0, -2.0, -3.0, -4....",nonstop
4,"[[0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0,...",nonstop
...,...,...
9995,"[[0.0, -1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0...",chill_stop
9996,"[[0.0, 1.0, 2.0, 2.0, 2.0, 3.0, 2.0, 2.0, 2.0,...",chill_stop
9997,"[[0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0...",nonstop
9998,"[[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 2.0, 2.0,...",nonstop


In [8]:
print(data.columns)

Index(['coords', 'movement_type'], dtype='object')


In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [10]:
valores_unicos = data['movement_type'].unique()
print(valores_unicos)

['hard_stop' 'nonstop' 'chill_stop']


In [11]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in data['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [12]:
# 2. Convertir las etiquetas a numéricas
le = LabelEncoder()
labels = le.fit_transform(data['movement_type'])
labels_tensor  = torch.tensor(labels, dtype=torch.long)
y_tensor = F.one_hot(labels_tensor, num_classes=3).float()

In [13]:
# 3. Reorganizar dimensiones para Conv1D (n_samples, channels, seq_length)
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [14]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [15]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [16]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length, num_classes):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        return c
    
# Antic
# class TrajectoryCNN(nn.Module):
#     def __init__(self, input_channels, seq_length, num_classes):
#         super().__init__()
#         self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
#         self.pool = nn.MaxPool1d(2)
        
#         # Cálculo dinámico de la dimensión final
#         self.fc_input_size = self._get_fc_size(seq_length, input_channels)
        
#         self.fc1 = nn.Linear(self.fc_input_size, 128)
#         self.fc2 = nn.Linear(128, num_classes)
    
#     def _get_fc_size(self, seq_length, channels):
#         dummy = torch.zeros(1, channels, seq_length)
#         dummy = self.pool(F.relu(self.conv1(dummy)))
#         dummy = self.pool(F.relu(self.conv2(dummy)))
#         return dummy.view(1, -1).size(1)
    
#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         x = x.view(x.size(0), -1)
#         x = F.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x


    # def forward(self, x):
    #     print(f"Input shape: {x.shape}")  # (batch_size, channels, seq_length)
        
    #     x = self.pool(F.relu(self.conv1(x)))
    #     print(f"After conv1 + relu + pool: {x.shape}")
        
    #     x = self.pool(F.relu(self.conv2(x)))
    #     print(f"After conv2 + relu + pool: {x.shape}")
        
    #     x = x.view(x.size(0), -1)
    #     print(f"After flatten: {x.shape}")
        
    #     x = F.relu(self.fc1(x))
    #     print(f"After fc1 + relu: {x.shape}")
        
    #     x = self.fc2(x)
    #     print(f"Output shape (logits): {x.shape}")
    
        # return x


In [17]:
# 7. Inicializar el modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)

In [18]:
model.first_conv.weight.shape

torch.Size([32, 2, 3])

In [19]:
# 8. Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [20]:
# 9. Entrenar el modelo
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)  # Cambiado a labels.argmax(dim=1)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 1.0922
Epoch [2/10], Loss: 1.0735
Epoch [3/10], Loss: 1.0318
Epoch [4/10], Loss: 0.4587
Epoch [5/10], Loss: 0.1566
Epoch [6/10], Loss: 0.1411
Epoch [7/10], Loss: 0.0523
Epoch [8/10], Loss: 0.0570
Epoch [9/10], Loss: 0.0470
Epoch [10/10], Loss: 0.0365


In [21]:
# 10. Evaluación (opcional)
with torch.no_grad():
    test_outputs = model(X_test)
    _, predicted = torch.max(test_outputs, 1)  # Índices de las clases predichas
    true_labels = torch.argmax(y_test, dim=1)  # Convertir one-hot a índices
    accuracy = (predicted == true_labels).float().mean()
    print(f'Test Accuracy: {accuracy.item():.4f}')

Test Accuracy: 0.9910


In [22]:
# Guardar solo los parámetros del modelo
torch.save(model.state_dict(), 'model_RandomWalk_movement_type_prediction.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'model_RandomWalk_movement_type_prediction_complete.pth')

In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()